# Génération du CSV de features

Pipeline masque + CLAHE appliqué image par image sur les 4 classes.
Les features numériques sont calculées sur l'image **après** CLAHE et masque.
`is_black` est la seule feature calculée sur l'image brute (avant CLAHE).

**Features :** `filename`, `classe`, `pixel_mean`, `pixel_std`, `pixel_median`,
`lum_interieur_masque`, `lum_exterieur_masque`, `surface_masque`, `variance_laplacien`, `is_black`


In [ ]:
import numpy as np
import pandas as pd
import cv2
import os
import fnmatch

# ── Chemin racine du dataset ──
base_path = r"C:\Users\Maxime\Desktop\Projet COVID19\COVID-19_Radiography_Dataset"

normal_path         = os.path.join(base_path, "Normal")
covid_path          = os.path.join(base_path, "COVID")
lungOpacity_path    = os.path.join(base_path, "Lung_Opacity")
viralPneumonia_path = os.path.join(base_path, "Viral Pneumonia")

# ── Chemin de sortie du CSV ──
output_csv = r"C:\Users\Maxime\Documents\Liora_Covid\data\processed\features.csv"


## 1. Comptage des images par classe


In [ ]:
nb_files_Normal         = len(fnmatch.filter(os.listdir(os.path.join(normal_path,         "images")), "*.png"))
nb_files_Covid          = len(fnmatch.filter(os.listdir(os.path.join(covid_path,          "images")), "*.png"))
nb_files_LungOpacity    = len(fnmatch.filter(os.listdir(os.path.join(lungOpacity_path,    "images")), "*.png"))
nb_files_ViralPneumonia = len(fnmatch.filter(os.listdir(os.path.join(viralPneumonia_path, "images")), "*.png"))

print('nb_files_Normal         :', nb_files_Normal)
print('nb_files_Covid          :', nb_files_Covid)
print('nb_files_LungOpacity    :', nb_files_LungOpacity)
print('nb_files_ViralPneumonia :', nb_files_ViralPneumonia)
print('Total                   :', nb_files_Normal + nb_files_Covid + nb_files_LungOpacity + nb_files_ViralPneumonia)


## 2. Pipeline masque + CLAHE + extraction des features

Ordre des opérations : chargement brut → détection image noire → CLAHE → application masque → features.


In [ ]:
def extract_features_for_class(class_path, class_label, img_prefix, nb_files):
    """Extrait les features de chaque image : pipeline masque puis CLAHE.
    Retourne une liste de dict prête à être convertie en DataFrame.
    """
    clahe = cv2.createCLAHE(clipLimit=10, tileGridSize=(8, 8))
    rows = []

    for i in range(nb_files):
        filename  = f"{img_prefix}-{i + 1}.png"
        img_path  = os.path.join(class_path, "images", filename)
        mask_path = os.path.join(class_path, "masks",  filename)

        img_raw  = cv2.imread(img_path,  cv2.IMREAD_GRAYSCALE)
        img_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if img_raw is None or img_mask is None:
            print(f"  [SKIP] {filename} — fichier manquant")
            continue

        # is_black calculé sur l'image brute avant tout traitement
        is_black = float(img_raw.mean()) < 5

        # redimensionnement masque 256x256 → 299x299 (INTER_NEAREST pour conserver le masque binaire)
        img_mask_resized = cv2.resize(
            img_mask,
            (img_raw.shape[1], img_raw.shape[0]),
            0, 0,
            cv2.INTER_NEAREST
        )
        mask_bin = img_mask_resized > 127  # seuil 127 : binarisation du masque

        # surface du masque (proportion pixels pulmonaires sur l'image resizée)
        surface_masque = float(mask_bin.sum()) / float(mask_bin.size)

        # CLAHE appliqué sur l'image brute, puis application du masque
        img_eq    = cv2.equalizeHist(img_raw)
        img_clahe = clahe.apply(img_eq)
        img_masked = cv2.bitwise_and(img_clahe, img_mask_resized)

        # toutes les features numériques sur l'image après CLAHE + masque
        pixel_mean   = float(img_masked.mean())
        pixel_std    = float(img_masked.std())
        pixel_median = float(np.median(img_masked))

        lum_interieur_masque = float(img_masked[mask_bin].mean())  if mask_bin.any()   else float('nan')
        lum_exterieur_masque = float(img_masked[~mask_bin].mean()) if (~mask_bin).any() else float('nan')

        variance_laplacien = float(cv2.Laplacian(img_masked, cv2.CV_64F).var())

        rows.append({
            'filename'             : filename,
            'classe'               : class_label,
            'pixel_mean'           : round(pixel_mean, 4),
            'pixel_std'            : round(pixel_std, 4),
            'pixel_median'         : pixel_median,
            'lum_interieur_masque' : round(lum_interieur_masque, 4),
            'lum_exterieur_masque' : round(lum_exterieur_masque, 4),
            'surface_masque'       : round(surface_masque, 6),
            'variance_laplacien'   : round(variance_laplacien, 4),
            'is_black'             : is_black,
        })

        if (i + 1) % 500 == 0:
            print(f"  {i + 1} / {nb_files} images traitées")

    print(f"  {len(rows)} images traitées — classe {class_label} terminée.")
    return rows


## 3. Extraction pour les 4 classes


In [ ]:
# Configuration des 4 classes
# (class_path, class_label_csv, img_prefix, nb_files)
# img_prefix : préfixe du nom de fichier tel que stocké dans le dataset
classes_config = [
    (covid_path,          'COVID',           'COVID',          nb_files_Covid),
    (normal_path,         'Normal',          'Normal',         nb_files_Normal),
    (lungOpacity_path,    'Lung_Opacity',    'Lung_Opacity',   nb_files_LungOpacity),
    (viralPneumonia_path, 'Viral_Pneumonia', 'Viral Pneumonia', nb_files_ViralPneumonia),
]

all_rows = []
for class_path, class_label, img_prefix, nb_files in classes_config:
    print(f"\n=== {class_label} ({nb_files} images) ===")
    rows = extract_features_for_class(class_path, class_label, img_prefix, nb_files)
    all_rows.extend(rows)

print(f"\nTotal lignes extraites : {len(all_rows)}")


## 4. Construction du DataFrame et sauvegarde


In [ ]:
# Création du dossier de sortie si nécessaire
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

df_features = pd.DataFrame(all_rows)

# Ordre des colonnes
df_features = df_features[[
    'filename', 'classe',
    'pixel_mean', 'pixel_std', 'pixel_median',
    'lum_interieur_masque', 'lum_exterieur_masque',
    'surface_masque', 'variance_laplacien', 'is_black',
]]

df_features.to_csv(output_csv, index=False)
print(f"CSV sauvegardé : {output_csv}")
print(f"Shape : {df_features.shape}")


## 5. Aperçu et statistiques descriptives


In [ ]:
print("── 5 premières lignes ──")
display(df_features.head())


In [ ]:
print("── Répartition par classe ──")
display(df_features['classe'].value_counts().to_frame('n_images'))


In [ ]:
print("── Statistiques descriptives (features numériques) ──")
display(df_features.describe())


In [ ]:
print("── Images noires détectées (is_black=True) ──")
display(df_features[df_features['is_black']].groupby('classe').size().rename('n_black').to_frame())


## 6. Visualisation du pipeline sur une image exemple


In [ ]:
import matplotlib.pyplot as plt

# ── Chargement COVID-1.png ──
img_path  = os.path.join(covid_path, "images", "COVID-1.png")
mask_path = os.path.join(covid_path, "masks",  "COVID-1.png")

img_raw  = cv2.imread(img_path,  cv2.IMREAD_GRAYSCALE)
img_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

# Redimensionnement masque 256x256 → 299x299 (INTER_NEAREST pour conserver le masque binaire)
img_mask_resized = cv2.resize(img_mask, (img_raw.shape[1], img_raw.shape[0]), 0, 0, cv2.INTER_NEAREST)

# Étape intermédiaire : masque appliqué sur image brute (pour montrer la segmentation)
img_avec_masque = cv2.bitwise_and(img_raw, img_mask_resized)

# Étape finale : CLAHE puis masque (pipeline complet)
clahe_viz  = cv2.createCLAHE(clipLimit=10, tileGridSize=(8, 8))
img_finale = cv2.bitwise_and(clahe_viz.apply(cv2.equalizeHist(img_raw)), img_mask_resized)

# ── Figure ──
output_fig = r"C:\Users\Maxime\Documents\Liora_Covid\reports\figures\exemple_pipeline.png"
os.makedirs(os.path.dirname(output_fig), exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(img_raw, cmap="gray")
axes[0].set_title("Image brute")
axes[0].axis("off")

axes[1].imshow(img_avec_masque, cmap="gray")
axes[1].set_title("Après application du masque")
axes[1].axis("off")

axes[2].imshow(img_finale, cmap="gray")
axes[2].set_title("Après CLAHE + masque")
axes[2].axis("off")

fig.suptitle("Pipeline pre-processing — COVID-1.png", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(output_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure sauvegardée : {output_fig}")
